# Lab 3 exercise solution - part 1

Exercise: *Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.*

This is the evaluator pattern used as a guardrail on the output. After the digital twin forms its reply, a second LLM acts as a judge and checks that the reply is strictly professional. If the reply is rejected, a loop sends the feedback back to the twin so it can refine its answer - a small piece of orchestration on top of the chat from lab 3.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

## Adapted from lab 3: my own twin profile and system prompt

The profile files live in the `twin` folder next to this notebook, and all three are loaded as context for the LLM:

- `cv.pdf` - my CV
- `linkedin.pdf` - my LinkedIn profile
- `sources.txt` - a combined text corpus with a personal summary and other source material

In [ ]:
def read_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text

cv = read_pdf("twin/cv.pdf")
linkedin = read_pdf("twin/linkedin.pdf")

with open("twin/sources.txt", "r", encoding="utf-8") as f:
    sources = f.read()

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{sources}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is the person's CV so that you can answer questions:

{cv}

Here is a summary of the person's LinkedIn profile:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

## New: the evaluator

A second LLM acts as a judge, like in lab 2. It sees the visitor's question and the twin's reply, and decides whether the reply sticks to professional matters. As in lab 2, the judge responds with JSON so we can parse its verdict in code.

In [ ]:
evaluator_system_prompt = """You are an evaluator acting as a guardrail for a digital twin chatting with visitors on a personal website.
The digital twin must only respond to professional questions pertaining to the person's career, background, skills and work experience.
It must not engage with personal topics such as food preferences, family, hobbies or private life;
for those it should politely decline and steer the conversation back to professional topics.

Given the visitor's question and the twin's reply, decide whether the reply is acceptable.
A polite refusal of a personal question is acceptable; engaging with the personal topic is not.

Respond with JSON, and only JSON, with the following format:
{"is_acceptable": true or false, "feedback": "brief feedback explaining your decision"}

Do not include markdown formatting or code blocks."""

In [ ]:
def evaluate(reply, message):
    evaluator_user_prompt = f"""Here is the visitor's question:

{message}

Here is the digital twin's reply:

{reply}

Now respond with the JSON verdict, nothing else."""

    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt}
    ]
    response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
    return json.loads(response.choices[0].message.content)

Let's test the evaluator on its own: a professional reply should pass, and a reply that engages with a personal question should be rejected.

In [ ]:
evaluate("I'm a Senior Engineering Manager in Stockholm, leading distributed software engineering teams with a focus on platform engineering and delivery.", "What do you do for a living?")

In [ ]:
evaluate("I love French food, but strangely I can't stand most cheese - except on cheesecake and pizza!", "What's your favorite food?")

## New: the refinement loop

Now we orchestrate the two LLMs. The twin forms a reply, the evaluator judges it, and if the reply is rejected we loop back: the twin gets its rejected attempt and the evaluator's feedback, and tries again. A retry limit stops us from looping forever.

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"""

# Previous reply rejected

You just tried to reply, but a quality control evaluator rejected your reply because it did not stick to professional matters.

Your attempted reply:
{reply}

Reason for rejection:
{feedback}

Please reply again. Politely decline to discuss personal topics and steer the conversation back to professional matters."""

    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message)
    attempts = 1
    while not evaluation["is_acceptable"] and attempts < 3:
        print(f"Attempt {attempts} rejected by the evaluator: {evaluation['feedback']}")
        reply = rerun(reply, message, history, evaluation["feedback"])
        evaluation = evaluate(reply, message)
        attempts += 1

    return reply

Let's try it: a professional question should pass the evaluator on the first attempt, and a personal question should either be declined immediately or refined in the loop. Watch the notebook output for the rejection messages.

In [ ]:
display(Markdown(chat("Tell me about your experience leading engineering teams.", [])))

In [ ]:
display(Markdown(chat("What's your favorite food? Do you like cheese?", [])))

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)